In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Costruire circuiti

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Raccomandiamo di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
```
</details>
Questa pagina approfondisce la classe [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) nell'SDK di Qiskit, inclusi alcuni metodi più avanzati che puoi usare per creare circuiti quantistici.
## Che cos'è un circuito quantistico?
Un semplice circuito quantistico è un insieme di qubit e un elenco di istruzioni che agiscono su quei qubit. Per illustrarlo, la cella seguente crea un nuovo circuito con due nuovi qubit, quindi visualizza l'attributo [`qubits`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qubits) del circuito, che è un elenco di [`Qubit`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Qubit) in ordine dal bit meno significativo $q_0$ al bit più significativo $q_n$.

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.qubits

[<Qubit register=(2, "q"), index=0>, <Qubit register=(2, "q"), index=1>]

Multiple `QuantumRegister` and `ClassicalRegister` objects can be combined to create a circuit. Every [`QuantumRegister`](/docs/api/qiskit/circuit#qiskit.circuit.QuantumRegister) and [`ClassicalRegister`](/docs/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) can also be named.

In [2]:
from qiskit.circuit import QuantumRegister, ClassicalRegister

qr1 = QuantumRegister(2, "qreg1")  # Create a QuantumRegister with 2 qubits
qr2 = QuantumRegister(1, "qreg2")  # Create a QuantumRegister with 1 qubit
cr1 = ClassicalRegister(3, "creg1")  # Create a ClassicalRegister with 3 cbits

combined_circ = QuantumCircuit(
    qr1, qr2, cr1
)  # Create a quantum circuit with 2 QuantumRegisters and 1 ClassicalRegister
combined_circ.qubits

[<Qubit register=(2, "qreg1"), index=0>,
 <Qubit register=(2, "qreg1"), index=1>,
 <Qubit register=(1, "qreg2"), index=0>]

È possibile combinare più oggetti `QuantumRegister` e `ClassicalRegister` per creare un circuito. Ogni [`QuantumRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.QuantumRegister) e [`ClassicalRegister`](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.ClassicalRegister) può anche essere denominato.

In [3]:
desired_qubit = qr2[0]  # Qubit 0 of register 'qreg2'

print("Index:", combined_circ.find_bit(desired_qubit).index)
print("Register:", combined_circ.find_bit(desired_qubit).registers)

Index: 2
Register: [(QuantumRegister(1, 'qreg2'), 0)]


Adding an instruction to the circuit appends the instruction to the circuit's [`data`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#data) attribute. The following cell output shows `data` is a list of [`CircuitInstruction`](/docs/api/qiskit/qiskit.circuit.CircuitInstruction) objects, each of which has an `operation` attribute, and a `qubits` attribute.

In [4]:
qc.x(0)  # Add X-gate to qubit 0
qc.data

[CircuitInstruction(operation=Instruction(name='x', num_qubits=1, num_clbits=0, params=[]), qubits=(<Qubit register=(2, "q"), index=0>,), clbits=())]

Puoi trovare l'indice e il registro di un qubit usando il metodo [`find_bit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.find_bit) del circuito e i suoi attributi.

In [5]:
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg" alt="Output of the previous code cell" />

Circuit instruction objects can contain "definition" circuits that describe the instruction in terms of more fundamental instructions. For example, the [X-gate](/docs/api/qiskit/qiskit.circuit.library.XGate) is defined as a specific case of the [U3-gate](/docs/api/qiskit/qiskit.circuit.library.U3Gate), a more general single-qubit gate.

In [6]:
# Draw definition circuit of 0th instruction in `qc`
qc.data[0].operation.definition.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg" alt="Output of the previous code cell" />

Aggiungere un'istruzione al circuito fa sì che l'istruzione venga accodata all'attributo [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#data) del circuito. L'output della cella seguente mostra che `data` è un elenco di oggetti [`CircuitInstruction`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.CircuitInstruction), ognuno dei quali ha un attributo `operation` e un attributo `qubits`.

In [7]:
from qiskit.circuit.library import HGate

qc = QuantumCircuit(1)
qc.append(
    HGate(),  # New HGate instruction
    [0],  # Apply to qubit 0
)
qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg" alt="Output of the previous code cell" />

To combine two circuits, use the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method. This accepts another [`QuantumCircuit`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit) and an optional list of qubit mappings.

<Admonition type="note">
    The [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method returns a new circuit and does **not** mutate either circuit it acts on. To mutate the circuit on which you're calling the [`compose`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#compose) method, use the argument `inplace=True`.
</Admonition>

In [8]:
qc_a = QuantumCircuit(4)
qc_a.x(0)

qc_b = QuantumCircuit(2, name="qc_b")
qc_b.y(0)
qc_b.z(1)

# compose qubits (0, 1) of qc_a to qubits (1, 3) of qc_b respectively
combined = qc_a.compose(qc_b, qubits=[1, 3])
combined.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/29152dfa-2275-4bc4-aadb-82185b9e0e86-0.svg" alt="Output of the previous code cell" />

Il modo più semplice per visualizzare queste informazioni è tramite il metodo [`draw`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#draw), che restituisce una rappresentazione grafica di un circuito. Consulta la pagina [Visualizzare i circuiti](./visualize-circuits) per i diversi modi di visualizzare i circuiti quantistici.

In [9]:
inst = qc_b.to_instruction()
qc_a.append(inst, [1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/81b682dd-45cb-4492-809e-d9e8ebbf5600-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/43a57258-3e33-4071-8a48-2bf127c8a5be-0.svg)

Gli oggetti istruzione del circuito possono contenere circuiti di "definizione" che descrivono l'istruzione in termini di istruzioni più fondamentali. Ad esempio, la [X-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.XGate) è definita come un caso particolare della [U3-gate](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.U3Gate), un gate a singolo qubit più generale.

In [10]:
gate = qc_b.to_gate().control()
qc_a.append(gate, [0, 1, 3])
qc_a.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/653e2427-e301-4d2f-84de-1959185ace8e-0.svg)

Istruzioni e circuiti si somigliano nel senso che entrambi descrivono operazioni su bit e qubit, ma hanno scopi diversi:

- Le istruzioni sono trattate come fisse, e i loro metodi di solito restituiscono nuove istruzioni (senza mutare l'oggetto originale).
- I circuiti sono progettati per essere costruiti in molte righe di codice, e i metodi di [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) spesso mutano l'oggetto esistente.
### Che cos'è la profondità di un circuito?
La [profondità (depth())](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.depth) di un circuito quantistico è una misura del numero di "strati" di gate quantistici, eseguiti in parallelo, necessari per completare il calcolo definito dal circuito. Poiché i gate quantistici richiedono tempo per essere implementati, la profondità di un circuito corrisponde approssimativamente al tempo che impiega il computer quantistico per eseguirlo. La profondità è quindi una delle quantità importanti usate per valutare se un circuito quantistico può essere eseguito su un dispositivo.

Il resto di questa pagina illustra come manipolare i circuiti quantistici.
## Costruire circuiti
Metodi come [`QuantumCircuit.h`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#h) e [`QuantumCircuit.cx`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#cx) aggiungono istruzioni specifiche ai circuiti. Per aggiungere istruzioni a un circuito in modo più generale, usa il metodo [`append`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#append). Questo metodo accetta un'istruzione e un elenco di qubit a cui applicarla. Consulta la [documentazione API della Circuit Library](https://docs.quantum.ibm.com/api/qiskit/circuit_library) per un elenco delle istruzioni supportate.

In [11]:
qc_a.decompose().draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/66813cae-9841-47ea-96b7-8fd7b82e9759-0.svg)

Per combinare due circuiti, usa il metodo [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose). Questo accetta un altro [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) e un elenco opzionale di mappature di qubit.

> **Note:** Il metodo [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose) restituisce un nuovo circuito e **non** muta nessuno dei circuiti su cui agisce. Per mutare il circuito su cui stai chiamando [`compose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#compose), usa l'argomento `inplace=True`.

In [12]:
qc1 = QuantumCircuit(2, 2)
qc1.measure(0, 1)
qc1.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/0cdb2273-0.svg" alt="Output of the previous code cell" />

In [13]:
qc2 = QuantumCircuit(2)
qc2.measure_all()
qc2.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/6f33698c-0.svg" alt="Output of the previous code cell" />

In [14]:
qc3 = QuantumCircuit(2)
qc3.x(1)
qc3.measure_active()
qc3.draw("mpl", cregbundle=False)

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/ed362e64-d6a4-4dfd-a5cf-5e6bdc7a81b5-0.svg)

Per capire cosa sta succedendo, puoi usare il metodo [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) per espandere ogni istruzione nella sua definizione.

> **Note:** Il metodo [`decompose`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#decompose) restituisce un nuovo circuito e **non** muta il circuito su cui agisce.

In [15]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit import Parameter

angle = Parameter("angle")  # undefined number

# Create and optimize circuit once
qc = QuantumCircuit(1)
qc.rx(angle, 0)
qc = generate_preset_pass_manager(
    optimization_level=3, basis_gates=["u", "cx"]
).run(qc)

qc.draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/a580552c-d585-4047-99f0-32aafd06e4f3-0.svg" alt="Output of the previous code cell" />

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/3c0633db-929b-4428-a888-7a3d493bd6dd-0.svg)

<span id="measure-qubits"></span>
## Misurare i qubit
Le misurazioni vengono usate per campionare gli stati dei singoli qubit e trasferire i risultati in un registro classico. Tieni presente che se stai inviando circuiti a una primitiva [Sampler](./primitives#sampler), le misurazioni sono obbligatorie. Invece, i circuiti inviati a una primitiva [Estimator](./primitives#estimator) non devono contenere misurazioni.

I qubit possono essere misurati usando tre metodi: [`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#qiskit.circuit.QuantumCircuit.measure), [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) e [`measure_active`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_active). Per scoprire come visualizzare i risultati delle misurazioni, consulta la pagina [Visualizzare i risultati](./visualize-results).

1. `QuantumCircuit.measure` : misura ogni qubit nel primo argomento sul bit classico fornito come secondo argomento. Questo metodo consente il pieno controllo su dove viene memorizzato il risultato della misurazione.

2. `QuantumCircuit.measure_all` : non richiede argomenti e può essere usato per circuiti quantistici senza bit classici predefiniti. Crea i fili classici e memorizza i risultati delle misurazioni in ordine. Ad esempio, la misurazione del qubit $q_i$ viene memorizzata nel cbit $meas_i$). Aggiunge anche una barriera prima della misurazione.

3. `QuantumCircuit.measure_active` : simile a `measure_all`, ma misura solo i qubit che hanno operazioni.

In [16]:
circuits = []
for value in range(100):
    circuits.append(qc.assign_parameters({angle: value}))

circuits[0].draw("mpl")

<Image src="../docs/images/guides/construct-circuits/extracted-outputs/85af6231-921a-4130-99d3-f6998f761df8-0.svg" alt="Output of the previous code cell" />

You can find a list of a circuit's undefined parameters in its `parameters` attribute.

In [17]:
qc.parameters

ParameterView([Parameter(angle)])

### Change a parameter's name

By default, parameter names for a parameterized circuit are prefixed by `x`- for example, `x[0]`. You can change the names after they are defined, as shown in the following example.

In [18]:
from qiskit.circuit.library import z_feature_map
from qiskit.circuit import ParameterVector

# Define a parameterized circuit with default names
# For example, x[0]
circuit = z_feature_map(2)

# Set new parameter names
# They will now be prefixed by `hi` instead
# For example, hi[0]
training_params = ParameterVector("hi", 2)

# Assign parameter names to the quantum circuit
circuit = circuit.assign_parameters(parameters=training_params)

![Output della cella di codice precedente](../docs/images/guides/construct-circuits/extracted-outputs/ca3f225f-0.svg)

## Circuiti parametrizzati

Molti algoritmi quantistici a breve termine prevedono l'esecuzione di numerose varianti di un circuito quantistico. Poiché la costruzione e l'ottimizzazione di circuiti di grandi dimensioni possono essere costose dal punto di vista computazionale, Qiskit supporta i circuiti **parametrizzati**. Questi circuiti hanno parametri non definiti, i cui valori non devono essere specificati fino a poco prima dell'esecuzione del circuito. Questo ti permette di spostare la costruzione e l'ottimizzazione del circuito fuori dal ciclo principale del programma. La cella seguente crea e visualizza un circuito parametrizzato.